# Flappy Bird RL - Fast Training with Live Dashboard

Train a Double DQN agent to play Flappy Bird with **16 parallel environments** and a **live updating dashboard**.

**Setup:** Runtime > Change runtime type > **T4 GPU** (optional — the real speedup comes from parallel envs)

In [ ]:
# Clone repo and install deps
!git clone https://github.com/tanmaysh17/flappy-bird.git 2>/dev/null || echo "Already cloned"
%cd flappy-bird
!pip install -q torch numpy matplotlib

In [ ]:
import torch, os, time, random, collections
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── Train with 16 parallel envs + live dashboard ──────────────────────
from flappy_rl.game import FlappyBirdEnv
from flappy_rl.agent import DQNAgent

NUM_ENVS = 16
NUM_EPISODES = 5000
PLOT_EVERY = 25        # update dashboard every N completed episodes

# Create parallel envs and single shared agent
envs = [FlappyBirdEnv() for _ in range(NUM_ENVS)]
agent = DQNAgent(device=device)
os.makedirs('checkpoints', exist_ok=True)

# Per-env state tracking
states = [env.reset() for env in envs]
active = [True] * NUM_ENVS

# Logging
all_scores = []
all_losses = []
all_epsilons = []
episodes_done = 0
t0 = time.time()

# Set up live plot
fig, axes = plt.subplots(2, 2, figsize=(14, 8), facecolor='#1a1a2e')
for ax in axes.flat:
    ax.set_facecolor('#16213e')
    ax.tick_params(colors='#8899aa')
    ax.spines['bottom'].set_color('#334455')
    ax.spines['left'].set_color('#334455')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
dh = display(fig, display_id=True)

def update_dashboard():
    for ax in axes.flat:
        ax.clear()
        ax.set_facecolor('#16213e')
        ax.tick_params(colors='#8899aa')
        ax.spines['bottom'].set_color('#334455')
        ax.spines['left'].set_color('#334455')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    elapsed = time.time() - t0
    eps_per_sec = len(all_scores) / max(elapsed, 1)
    fig.suptitle(
        f'Flappy Bird RL  |  {len(all_scores)}/{NUM_EPISODES} eps  |  '
        f'{eps_per_sec:.1f} ep/s  |  {elapsed:.0f}s  |  '
        f'Best: {max(all_scores) if all_scores else 0}  |  '
        f'Avg100: {np.mean(all_scores[-100:]):.1f if all_scores else 0}',
        color='white', fontsize=13, fontweight='bold'
    )

    # Score chart
    ax = axes[0, 0]
    ax.plot(all_scores, alpha=0.4, color='#00d2ff', linewidth=0.5)
    if len(all_scores) >= 50:
        avg = np.convolve(all_scores, np.ones(50)/50, mode='valid')
        ax.plot(range(49, 49+len(avg)), avg, color='#ff6b35', linewidth=2)
    ax.set_title('Score per Episode', color='white', fontsize=11)
    ax.set_ylabel('Score', color='#8899aa')
    ax.fill_between(range(len(all_scores)), all_scores, alpha=0.15, color='#00d2ff')

    # Rolling average
    ax = axes[0, 1]
    if len(all_scores) >= 20:
        avgs = [np.mean(all_scores[max(0,i-20):i+1]) for i in range(len(all_scores))]
        ax.plot(avgs, color='#ff6b35', linewidth=2)
        ax.fill_between(range(len(avgs)), avgs, alpha=0.2, color='#ff6b35')
    ax.set_title('Rolling Avg (20 eps)', color='white', fontsize=11)
    ax.set_ylabel('Avg Score', color='#8899aa')

    # Loss chart (log scale)
    ax = axes[1, 0]
    if all_losses:
        positive = [(i, l) for i, l in enumerate(all_losses) if l > 0]
        if positive:
            ax.semilogy([x[0] for x in positive], [x[1] for x in positive],
                       color='#a855f7', linewidth=1, alpha=0.7)
    ax.set_title('Loss (log scale)', color='white', fontsize=11)
    ax.set_ylabel('Loss', color='#8899aa')
    ax.set_xlabel('Episode', color='#8899aa')

    # Epsilon chart
    ax = axes[1, 1]
    ax.plot(all_epsilons, color='#22c55e', linewidth=2)
    ax.fill_between(range(len(all_epsilons)), all_epsilons, alpha=0.2, color='#22c55e')
    ax.set_title('Epsilon (exploration)', color='white', fontsize=11)
    ax.set_ylabel('Epsilon', color='#8899aa')
    ax.set_xlabel('Episode', color='#8899aa')

    fig.tight_layout(rect=[0, 0, 1, 0.94])
    dh.update(fig)

# ── Main training loop with parallel envs ──────────────────────────
ep_losses = [[] for _ in range(NUM_ENVS)]
last_plot = 0

while episodes_done < NUM_EPISODES:
    # Step all active envs
    for i in range(NUM_ENVS):
        action = agent.select_action(states[i], training=True)
        ns, r, done = envs[i].step(action)
        agent.store_transition(states[i], action, r, ns, done)

        loss = agent.train_step()
        if loss is not None:
            ep_losses[i].append(loss)

        if done:
            all_scores.append(envs[i].score)
            all_losses.append(np.mean(ep_losses[i]) if ep_losses[i] else 0.0)
            all_epsilons.append(agent.epsilon)
            ep_losses[i] = []
            episodes_done += 1

            # Checkpoint
            if episodes_done % 1000 == 0:
                agent.save(f'checkpoints/agent_ep{episodes_done}.pt')

            # Reset this env
            states[i] = envs[i].reset()
        else:
            states[i] = ns

    # Update dashboard periodically
    if episodes_done - last_plot >= PLOT_EVERY:
        last_plot = episodes_done
        update_dashboard()

# Final save and dashboard update
agent.save('checkpoints/agent_final.pt')
update_dashboard()

elapsed = time.time() - t0
print(f'\nDone! {NUM_EPISODES} episodes in {elapsed:.0f}s ({NUM_EPISODES/elapsed:.1f} ep/s)')
print(f'Best: {max(all_scores)} | Avg(last 100): {np.mean(all_scores[-100:]):.1f}')

In [ ]:
# Final score distribution
fig2, ax = plt.subplots(figsize=(10, 4), facecolor='#1a1a2e')
ax.set_facecolor('#16213e')
ax.hist(all_scores[-500:], bins=30, color='#00d2ff', alpha=0.7, edgecolor='white')
ax.axvline(np.mean(all_scores[-500:]), color='#ff6b35', linewidth=2,
           label=f'Mean: {np.mean(all_scores[-500:]):.1f}')
ax.axvline(max(all_scores), color='#22c55e', linewidth=2, linestyle='--',
           label=f'Best: {max(all_scores)}')
ax.set_title('Score Distribution (last 500 episodes)', color='white', fontsize=13)
ax.set_xlabel('Score', color='#8899aa')
ax.legend(facecolor='#16213e', edgecolor='#334455', labelcolor='white')
ax.tick_params(colors='#8899aa')
plt.tight_layout()
plt.show()

In [ ]:
# Download the trained model
from google.colab import files
files.download('checkpoints/agent_final.pt')